# Filter YOLO Dataset - Remove Unwanted Classes

This notebook helps you remove specific classes from your YOLO dataset.

In [1]:
import os
import shutil
import yaml
from pathlib import Path

## Step 1: Check Current Classes in Dataset

In [2]:
# Load and display current classes
dataset_path = Path('../images/hazard detection.v1i.yolov11')
yaml_path = dataset_path / 'data.yaml'

with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print("Current classes in dataset:")
for i, class_name in enumerate(data_config['names']):
    print(f"  {i}: {class_name}")

print(f"\nTotal classes: {len(data_config['names'])}")

Current classes in dataset:
  0: knife
  1: non-violence
  2: violence
  3: weapon

Total classes: 4


## Step 2: Specify Classes to Remove

In [3]:
# List the classes you want to REMOVE
classes_to_remove = [
    'non-violence',  # Replace with actual class names you want to remove
    # 'class_name_2',
]

print("Classes marked for removal:")
for class_name in classes_to_remove:
    if class_name in data_config['names']:
        idx = data_config['names'].index(class_name)
        print(f"  ✓ {class_name} (index {idx})")
    else:
        print(f"  ✗ {class_name} - NOT FOUND!")

Classes marked for removal:
  ✓ non-violence (index 1)


## Step 3: Filter the Dataset

In [4]:
def filter_yolo_dataset(
    dataset_path: Path,
    classes_to_remove: list,
    output_path: Path = None,
    remove_empty_images: bool = True
):
    """
    Filter out specific classes from YOLO dataset
    """
    # Read data.yaml
    yaml_path = dataset_path / 'data.yaml'
    with open(yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)

    class_names = data_config['names']

    # Find indices to remove
    remove_indices = []
    for class_name in classes_to_remove:
        if class_name in class_names:
            remove_indices.append(class_names.index(class_name))

    if not remove_indices:
        print("❌ No valid classes to remove!")
        return

    # Create new class list and index mapping
    new_class_names = [name for i, name in enumerate(class_names) if i not in remove_indices]
    old_to_new_index = {}
    new_index = 0
    for old_index in range(len(class_names)):
        if old_index not in remove_indices:
            old_to_new_index[old_index] = new_index
            new_index += 1

    print(f"📊 Original: {len(class_names)} classes")
    print(f"📊 New: {len(new_class_names)} classes")
    print(f"\n🗑️  Removing: {[class_names[i] for i in remove_indices]}")
    print(f"✓ Keeping: {new_class_names}\n")

    # Setup output
    if output_path is None:
        output_path = dataset_path
    else:
        output_path.mkdir(parents=True, exist_ok=True)

    # Process each split
    total_stats = {'total': 0, 'filtered': 0, 'empty': 0, 'removed': 0}

    for split in ['train', 'valid', 'test']:
        images_dir = dataset_path / split / 'images'
        labels_dir = dataset_path / split / 'labels'

        if not labels_dir.exists():
            continue

        if output_path != dataset_path:
            output_images_dir = output_path / split / 'images'
            output_labels_dir = output_path / split / 'labels'
            output_images_dir.mkdir(parents=True, exist_ok=True)
            output_labels_dir.mkdir(parents=True, exist_ok=True)
        else:
            output_images_dir = images_dir
            output_labels_dir = labels_dir

        stats = {'total': 0, 'filtered': 0, 'removed': 0, 'empty': 0}

        for label_file in labels_dir.glob('*.txt'):
            stats['total'] += 1

            with open(label_file, 'r') as f:
                lines = f.readlines()

            # Filter annotations
            filtered_lines = []
            for line in lines:
                parts = line.strip().split()
                if not parts:
                    continue

                class_id = int(parts[0])

                if class_id in remove_indices:
                    stats['removed'] += 1
                    continue

                new_class_id = old_to_new_index[class_id]
                parts[0] = str(new_class_id)
                filtered_lines.append(' '.join(parts) + '\n')

            # Find corresponding image
            image_file = images_dir / (label_file.stem + '.jpg')
            if not image_file.exists():
                image_file = images_dir / (label_file.stem + '.png')

            if filtered_lines:
                output_label = output_labels_dir / label_file.name
                with open(output_label, 'w') as f:
                    f.writelines(filtered_lines)

                if output_path != dataset_path and image_file.exists():
                    shutil.copy2(image_file, output_images_dir / image_file.name)

                stats['filtered'] += 1
            else:
                stats['empty'] += 1

                if remove_empty_images and output_path == dataset_path:
                    label_file.unlink()
                    if image_file.exists():
                        image_file.unlink()

        print(f"📁 {split.upper()}: {stats['total']} files → "
              f"Kept: {stats['filtered']}, Empty: {stats['empty']}, "
              f"Annotations removed: {stats['removed']}")

        for key in total_stats:
            total_stats[key] += stats[key]

    # Update data.yaml
    data_config['names'] = new_class_names
    data_config['nc'] = len(new_class_names)

    output_yaml = output_path / 'data.yaml'
    with open(output_yaml, 'w') as f:
        yaml.dump(data_config, f, default_flow_style=False)

    print(f"\n✅ COMPLETE!")
    print(f"   Total files: {total_stats['total']}")
    print(f"   Files kept: {total_stats['filtered']}")
    print(f"   Empty files: {total_stats['empty']}")
    print(f"   Annotations removed: {total_stats['removed']}")
    print(f"   Classes: {len(class_names)} → {len(new_class_names)}")


# Run the filter
filter_yolo_dataset(
    dataset_path=dataset_path,
    classes_to_remove=classes_to_remove,
    output_path=None,  # Set to Path('new_dataset_path') to create new filtered dataset
    remove_empty_images=True  # Remove images with no annotations
)

📊 Original: 4 classes
📊 New: 3 classes

🗑️  Removing: ['non-violence']
✓ Keeping: ['knife', 'violence', 'weapon']

📁 TRAIN: 6654 files → Kept: 6593, Empty: 61, Annotations removed: 8406
📁 VALID: 1877 files → Kept: 1861, Empty: 16, Annotations removed: 2361
📁 TEST: 946 files → Kept: 939, Empty: 7, Annotations removed: 1165

✅ COMPLETE!
   Total files: 9477
   Files kept: 9393
   Empty files: 84
   Annotations removed: 11932
   Classes: 4 → 3


## Step 4: Verify the Filtered Dataset

In [5]:
# Check the updated classes
with open(yaml_path, 'r') as f:
    updated_config = yaml.safe_load(f)

print("Updated classes:")
for i, class_name in enumerate(updated_config['names']):
    print(f"  {i}: {class_name}")

print(f"\nTotal classes: {len(updated_config['names'])}")

Updated classes:
  0: knife
  1: violence
  2: weapon

Total classes: 3
